# Taller 02 — Actor-Critic con componente de descubrimiento (DAC) en Asteroids

**Entorno:** `ALE/Asteroids-v5` (Gymnasium + ALE), 14 acciones discretas, observación 84×84×4 (grises, 4 frames apilados).

**Idea:** al actor-critic clásico se le agregan dos conjuntos de redes:

1. **Ensamble de críticos** $V^E_1..V^E_K$: su desacuerdo $\sigma_V(s)$ mide cuánto se puede confiar en el crítico.
   - El gradiente del actor se pondera con $w(s)=1/(1+\kappa\,\tilde\sigma_V(s))$ → el crítico no "engaña" al actor donde no sabe.
2. **Módulo de descubrimiento**: ensamble de $M$ modelos de dinámica $g_m(\psi(s),a)\to\psi(s')$ sobre un codificador aleatorio fijo $\psi$.
   - **Recompensa intrínseca:** $r^I_t = \frac{1}{d}\sum_j \mathrm{Var}_m[g_m(\psi(s_t),a_t)]_j$ (novedad epistémica), con su propio crítico $V^I$.
   - **"Preguntar por otra acción":** cuando $\sigma_V(s_t)$ supera su media móvil (o con prob. $\epsilon$), el agente consulta el mapa de novedad $U(s,\cdot)$ de **todas** las acciones y actúa con
     $\mu(a|s)\propto \pi_\theta(a|s)\,\exp(\eta_t\,\hat U(s,a))$.
   - Como se actúa con $\mu\neq\pi$, se corrige con $\bar\rho_t=\min(1,\pi_\theta(a_t|s_t)/b(a_t|s_t))$.

**Pérdidas**

$$\mathcal L_\pi=-\mathbb E\big[\bar\rho_t\,w(s_t)\,\hat A_t\log\pi_\theta(a_t|s_t)\big]-\beta\,\mathcal H(\pi_\theta),\qquad \hat A_t=A^E_t+c_I A^I_t$$

$$\mathcal L_V=\sum_k \mathbb E\big[m_{k,t}(V^E_k(s_t)-R^E_t)^2\big]+\mathbb E\big[(V^I(s_t)-R^I_t)^2\big],\qquad \mathcal L_D=\sum_m\mathbb E\big[m_{m,t}\|g_m(\psi(s_t),a_t)-\psi(s_{t+1})\|^2\big]$$

$A^E, A^I$ con GAE($\lambda$) (la intrínseca es no episódica); $m$ son máscaras bootstrap Bernoulli que mantienen diversos los ensambles.

> Entorno de ejecución recomendado: **GPU** (Entorno de ejecución → Cambiar tipo → T4).

## 0. Instalación

In [ ]:
!pip install -q "gymnasium[atari]>=1.1" "ale-py>=0.10" imageio imageio-ffmpeg

## 1. Imports y entorno

In [ ]:
# ============================================================
#  DAC: Discovery Actor-Critic  (Asteroids - Gymnasium / ALE)
# ============================================================
import os, time, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Dispositivo:', DEVICE)

# 1. Entorno
# ------------------------------------------------------------
def make_env(env_id="ALE/Asteroids-v5", seed=0, render_mode=None):
    def thunk():
        # frameskip=1 en el entorno base: AtariPreprocessing hace el frame-skip (4)
        env = gym.make(env_id, frameskip=1, repeat_action_probability=0.25,
                       render_mode=render_mode)
        env = gym.wrappers.AtariPreprocessing(
            env, noop_max=30, frame_skip=4, screen_size=84,
            grayscale_obs=True, scale_obs=False, terminal_on_life_loss=False)
        env = gym.wrappers.FrameStackObservation(env, stack_size=4)
        env.reset(seed=seed)
        env.action_space.seed(seed)
        return env
    return thunk


def make_vec_env(env_id, num_envs, seed, asynchronous=True):
    fns = [make_env(env_id, seed + i) for i in range(num_envs)]
    kw = dict(autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
    venv = gym.vector.AsyncVectorEnv(fns, **kw) if asynchronous else gym.vector.SyncVectorEnv(fns, **kw)
    venv = gym.wrappers.vector.RecordEpisodeStatistics(venv)
    return venv

## 2. Utilidades

In [ ]:
# # 2. Utilidades
# ------------------------------------------------------------
class RunningMeanStd:
    """Media/varianza móviles (algoritmo paralelo de Chan)."""
    def __init__(self, shape=()):
        self.mean = np.zeros(shape, np.float64)
        self.var = np.ones(shape, np.float64)
        self.count = 1e-4

    def update(self, x):
        x = np.asarray(x, np.float64)
        b_mean, b_var, b_count = x.mean(0), x.var(0), x.shape[0]
        delta = b_mean - self.mean
        tot = self.count + b_count
        self.mean = self.mean + delta * b_count / tot
        m2 = self.var * self.count + b_var * b_count + delta ** 2 * self.count * b_count / tot
        self.var = m2 / tot
        self.count = tot


def nature_cnn(in_ch=4, out=512):
    return nn.Sequential(
        nn.Conv2d(in_ch, 32, 8, 4), nn.ReLU(),
        nn.Conv2d(32, 64, 4, 2), nn.ReLU(),
        nn.Conv2d(64, 64, 3, 1), nn.ReLU(),
        nn.Flatten(), nn.Linear(64 * 7 * 7, out), nn.ReLU())


def ortho_init(m, gain=math.sqrt(2)):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.orthogonal_(m.weight, gain)
        nn.init.zeros_(m.bias)

## 3. Actor + ensamble de críticos

In [ ]:
# # 3. Actor + ensamble de críticos
# ------------------------------------------------------------
class ActorCriticEnsemble(nn.Module):
    """
    Encoder compartido φ(s)
      Actor            π_θ(a|s)
      K críticos       V_k^E(s), k=1..K  (recompensa extrínseca)
      Crítico intrín.  V^I(s)            (recompensa de descubrimiento)
    """
    def __init__(self, n_actions, n_critics=3):
        super().__init__()
        self.encoder = nature_cnn()
        self.actor = nn.Linear(512, n_actions)
        self.critics = nn.ModuleList([nn.Linear(512, 1) for _ in range(n_critics)])
        self.critic_int = nn.Linear(512, 1)
        self.apply(ortho_init)
        ortho_init(self.actor, 0.01)
        for c in self.critics:        # cada cabeza con su propia inicialización
            ortho_init(c, 1.0)
        ortho_init(self.critic_int, 1.0)

    def forward(self, obs):
        h = self.encoder(obs.float() / 255.0)
        logits = self.actor(h)
        v_e = torch.cat([c(h) for c in self.critics], dim=1)   # (B, K)
        v_i = self.critic_int(h).squeeze(1)                    # (B,)
        return logits, v_e, v_i

## 4. Componente de descubrimiento

In [ ]:
# # 4. Componente de descubrimiento
# ------------------------------------------------------------
class DiscoveryModule(nn.Module):
    """
    ψ(s): codificador aleatorio FIJO (no entrenado) -> espacio de características estable.
    g_m(ψ(s), a) -> ψ(s'),  m = 1..M : ensamble de modelos de dinámica.
    El desacuerdo entre los M modelos mide qué tan poco conocida es la
    consecuencia de una acción ("novedad epistémica").
    """
    def __init__(self, n_actions, n_models=5, feat=256, hidden=256):
        super().__init__()
        self.n_actions = n_actions
        self.target = nn.Sequential(
            nn.Conv2d(4, 32, 8, 4), nn.LeakyReLU(),
            nn.Conv2d(32, 64, 4, 2), nn.LeakyReLU(),
            nn.Conv2d(64, 64, 3, 1), nn.LeakyReLU(),
            nn.Flatten(), nn.Linear(64 * 7 * 7, feat))
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.models = nn.ModuleList([
            nn.Sequential(nn.Linear(feat + n_actions, hidden), nn.ReLU(),
                          nn.Linear(hidden, hidden), nn.ReLU(),
                          nn.Linear(hidden, feat))
            for _ in range(n_models)])

    @torch.no_grad()
    def features(self, obs):
        f = self.target(obs.float() / 255.0)
        return f / (f.norm(dim=1, keepdim=True) + 1e-8) * math.sqrt(f.shape[1])

    def predict(self, feat, actions):
        a = F.one_hot(actions, self.n_actions).float()
        x = torch.cat([feat, a], dim=1)
        return torch.stack([m(x) for m in self.models], dim=0)   # (M, B, feat)

    @torch.no_grad()
    def disagreement(self, feat, actions):
        """D(s,a) = (1/d) Σ_j Var_m[g_m(ψ(s),a)]_j"""
        return self.predict(feat, actions).var(dim=0).mean(dim=1)

    @torch.no_grad()
    def novelty_map(self, feat):
        """U(s,a) para TODAS las acciones -> (B, A). Es la 'pregunta':
        ¿qué otra acción produciría consecuencias que aún no conozco?"""
        B, A = feat.shape[0], self.n_actions
        f = feat.unsqueeze(1).expand(B, A, -1).reshape(B * A, -1)
        a = torch.arange(A, device=feat.device).repeat(B)
        return self.disagreement(f, a).view(B, A)

## 5. Configuración (hiperparámetros)

In [ ]:
# # 5. Configuración
# ------------------------------------------------------------
def default_config(**kw):
    cfg = dict(
        env_id="ALE/Asteroids-v5", seed=1, num_envs=16, n_steps=16,
        total_steps=2_000_000,          # pasos de agente (x4 frames)
        lr=2.5e-4, gamma=0.99, gamma_i=0.99, lam=0.95,
        ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
        # --- ensamble de críticos ---
        n_critics=3, critic_mask_p=0.8, kappa=1.0,
        # --- descubrimiento ---
        use_discovery=True, n_models=5, int_coef=0.5, disc_lr=1e-4, disc_mask_p=0.8,
        eta0=1.0, eta_final=0.1,         # fuerza de la inclinación hacia acciones novedosas
        ask_eps=0.05,                    # probabilidad mínima de "preguntar"
        log_every=10, run_name="dac",
    )
    cfg.update(kw)
    return cfg


def baseline_config(**kw):
    """A2C estándar (un solo crítico, sin descubrimiento) para comparar."""
    return default_config(n_critics=1, kappa=0.0, critic_mask_p=1.0,
                          use_discovery=False, run_name="a2c_baseline", **kw)

## 6. Bucle de entrenamiento

In [ ]:
# # 6. Entrenamiento
# ------------------------------------------------------------
def compute_gae(rewards, values, dones, last_value, gamma, lam):
    """rewards, values, dones: (T, N). dones[t]=1 si el episodio terminó en t."""
    T = rewards.shape[0]
    adv = torch.zeros_like(rewards)
    gae = torch.zeros_like(last_value)
    for t in reversed(range(T)):
        nv = last_value if t == T - 1 else values[t + 1]
        nonterm = 1.0 - dones[t]
        delta = rewards[t] + gamma * nv * nonterm - values[t]
        gae = delta + gamma * lam * nonterm * gae
        adv[t] = gae
    return adv, adv + values


def train(cfg, save_dir="checkpoints", verbose=True):
    os.makedirs(save_dir, exist_ok=True)
    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    N, T = cfg["num_envs"], cfg["n_steps"]
    envs = make_vec_env(cfg["env_id"], N, cfg["seed"])
    A = envs.single_action_space.n
    K = cfg["n_critics"]

    net = ActorCriticEnsemble(A, K).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], eps=1e-5)
    disc = disc_opt = None
    if cfg["use_discovery"]:
        disc = DiscoveryModule(A, cfg["n_models"]).to(DEVICE)
        disc_opt = torch.optim.Adam(disc.models.parameters(), lr=cfg["disc_lr"])

    int_ret_rms = RunningMeanStd()
    int_ret_running = np.zeros(N)
    sigma_ema = None                         # media móvil del desacuerdo de los críticos

    obs_shape = envs.single_observation_space.shape
    b_obs = torch.zeros((T, N) + obs_shape, dtype=torch.uint8, device=DEVICE)
    b_next = torch.zeros_like(b_obs)
    b_act = torch.zeros((T, N), dtype=torch.long, device=DEVICE)
    b_logb = torch.zeros((T, N), device=DEVICE)      # log b(a|s) de la política de comportamiento
    b_rew = torch.zeros((T, N), device=DEVICE)
    b_done = torch.zeros((T, N), device=DEVICE)
    b_ask = torch.zeros((T, N), device=DEVICE)

    obs, _ = envs.reset(seed=cfg["seed"])
    obs = torch.as_tensor(obs, device=DEVICE)
    n_updates = cfg["total_steps"] // (N * T)
    log, ep_returns, global_step, t0 = [], [], 0, time.time()

    for update in range(1, n_updates + 1):
        frac = (update - 1) / n_updates
        for g in opt.param_groups:
            g["lr"] = cfg["lr"] * (1 - frac)
        eta = cfg["eta0"] + (cfg["eta_final"] - cfg["eta0"]) * frac

        # ---------------- recolección ----------------
        for t in range(T):
            with torch.no_grad():
                logits, v_e, _ = net(obs)
                logp_pi = F.log_softmax(logits, -1)
                if disc is not None:
                    # ¿Debe preguntar? -> cuando los críticos no se ponen de acuerdo
                    sigma = v_e.std(1) if K > 1 else torch.zeros(N, device=DEVICE)
                    sm = sigma.mean().item()
                    sigma_ema = sm if sigma_ema is None else 0.99 * sigma_ema + 0.01 * sm
                    ask = (sigma > sigma_ema) | (torch.rand(N, device=DEVICE) < cfg["ask_eps"])
                    U = disc.novelty_map(disc.features(obs))
                    U = (U - U.mean(1, keepdim=True)) / (U.std(1, keepdim=True) + 1e-8)
                    logp_mu = F.log_softmax(logp_pi + eta * U, -1)
                    logp_b = torch.where(ask.unsqueeze(1), logp_mu, logp_pi)
                else:
                    ask = torch.zeros(N, dtype=torch.bool, device=DEVICE)
                    logp_b = logp_pi
                action = torch.distributions.Categorical(logits=logp_b).sample()

            nobs, rew, term, trunc, info = envs.step(action.cpu().numpy())
            done = np.logical_or(term, trunc)
            # observación siguiente real (antes del reset automático)
            real_next = nobs.copy()
            if "final_obs" in info:
                for i in np.where(info.get("_final_obs", done))[0]:
                    real_next[i] = info["final_obs"][i]
            if "episode" in info:
                for i in np.where(info["_episode"])[0]:
                    ep_returns.append((global_step + (t + 1) * N, float(info["episode"]["r"][i])))

            b_obs[t] = obs; b_next[t] = torch.as_tensor(real_next, device=DEVICE)
            b_act[t] = action; b_logb[t] = logp_b.gather(1, action[:, None]).squeeze(1)
            b_rew[t] = torch.as_tensor(np.sign(rew), device=DEVICE, dtype=torch.float32)
            b_done[t] = torch.as_tensor(done, device=DEVICE, dtype=torch.float32)
            b_ask[t] = ask.float()
            obs = torch.as_tensor(nobs, device=DEVICE)
        global_step += N * T

        # ---------------- recompensa intrínseca ----------------
        flat_obs = b_obs.reshape((-1,) + obs_shape)
        flat_next = b_next.reshape((-1,) + obs_shape)
        flat_act = b_act.reshape(-1)
        if disc is not None:
            f_s, f_n = disc.features(flat_obs), disc.features(flat_next)
            r_int = disc.disagreement(f_s, flat_act).view(T, N).cpu().numpy()
            # normalización por la desviación del retorno intrínseco descontado (estilo RND)
            rets = []
            for t in range(T):
                int_ret_running = int_ret_running * cfg["gamma_i"] + r_int[t]
                rets.append(int_ret_running.copy())
            int_ret_rms.update(np.concatenate(rets))
            r_int = torch.as_tensor(r_int / math.sqrt(int_ret_rms.var + 1e-8),
                                    device=DEVICE, dtype=torch.float32)
        else:
            r_int = torch.zeros((T, N), device=DEVICE)

        # ---------------- ventajas ----------------
        with torch.no_grad():
            logits_all, ve_all, vi_all = net(flat_obs)
            ve_mean = ve_all.mean(1).view(T, N)
            vi_all = vi_all.view(T, N)
            _, ve_last, vi_last = net(obs)
            adv_e, ret_e = compute_gae(b_rew, ve_mean, b_done, ve_last.mean(1),
                                       cfg["gamma"], cfg["lam"])
            # intrínseca: no episódica (la curiosidad no muere con el episodio)
            adv_i, ret_i = compute_gae(r_int, vi_all, torch.zeros_like(b_done), vi_last,
                                       cfg["gamma_i"], cfg["lam"])
        adv = adv_e + (cfg["int_coef"] * adv_i if disc is not None else 0.0)

        # ---------------- pérdidas ----------------
        logits_all, ve_all, vi_all = net(flat_obs)
        logp = F.log_softmax(logits_all, -1)
        logp_a = logp.gather(1, flat_act[:, None]).squeeze(1)
        # corrección off-policy: ρ = min(1, π/b)
        rho = torch.exp(logp_a.detach() - b_logb.reshape(-1)).clamp(max=1.0)
        # confianza en el crítico: w = 1 / (1 + κ σ̃_V)
        if K > 1:
            sig = ve_all.detach().std(1)
            w = 1.0 / (1.0 + cfg["kappa"] * sig / (sig.mean() + 1e-8))
        else:
            sig = torch.zeros_like(logp_a); w = torch.ones_like(logp_a)
        a_flat = adv.reshape(-1)
        a_flat = (a_flat - a_flat.mean()) / (a_flat.std() + 1e-8)
        pg_loss = -(rho * w * a_flat * logp_a).mean()
        entropy = -(logp.exp() * logp).sum(1).mean()
        # cada crítico se entrena con una máscara bootstrap distinta -> diversidad
        mask = (torch.rand_like(ve_all) < cfg["critic_mask_p"]).float()
        v_loss_e = (((ve_all - ret_e.reshape(-1, 1)) ** 2) * mask).sum() / mask.sum().clamp(min=1)
        v_loss_i = ((vi_all - ret_i.reshape(-1)) ** 2).mean() if disc is not None else torch.tensor(0.0, device=DEVICE)
        loss = pg_loss - cfg["ent_coef"] * entropy + cfg["vf_coef"] * (v_loss_e + v_loss_i)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), cfg["max_grad_norm"])
        opt.step()

        disc_loss = 0.0
        if disc is not None:
            valid = (1.0 - b_done.reshape(-1))         # no aprender transiciones a través de un reset
            pred = disc.predict(f_s, flat_act)          # (M, B, d)
            err = ((pred - f_n.unsqueeze(0)) ** 2).mean(2)
            m = (torch.rand_like(err) < cfg["disc_mask_p"]).float() * valid.unsqueeze(0)
            dl = (err * m).sum() / m.sum().clamp(min=1)
            disc_opt.zero_grad(); dl.backward(); disc_opt.step()
            disc_loss = dl.item()

        # ---------------- registro ----------------
        if update % cfg["log_every"] == 0 or update == n_updates:
            recent = [r for _, r in ep_returns[-50:]]
            row = dict(step=global_step, fps=int(global_step / (time.time() - t0)),
                       score_mean=float(np.mean(recent)) if recent else float("nan"),
                       pg_loss=pg_loss.item(), v_loss=v_loss_e.item(),
                       v_loss_int=float(v_loss_i.item()), disc_loss=disc_loss,
                       entropy=entropy.item(), critic_std=float(sig.mean().item()),
                       r_int=float(r_int.mean().item()), ask_rate=float(b_ask.mean().item()),
                       rho=float(rho.mean().item()), eta=eta)
            log.append(row)
            if verbose:
                print(f"[{cfg['run_name']}] step {global_step:>9,} | score {row['score_mean']:7.1f} | "
                      f"H {row['entropy']:.2f} | σV {row['critic_std']:.3f} | r_int {row['r_int']:.3f} | "
                      f"ask {row['ask_rate']:.2f} | fps {row['fps']}")

    envs.close()
    path = os.path.join(save_dir, f"{cfg['run_name']}.pt")
    torch.save(dict(cfg=cfg, n_actions=A, net=net.state_dict(),
                    disc=disc.state_dict() if disc is not None else None), path)
    with open(os.path.join(save_dir, f"{cfg['run_name']}_log.json"), "w") as f:
        json.dump(dict(log=log, episodes=ep_returns), f)
    if verbose:
        print("Modelo guardado en", path)
    return net, disc, log, ep_returns